# Pengujian Sistem Machine Learning - NEU-DET Classifier

Notebook ini melakukan **prediction request** ke endpoint cloud yang sudah dideploy dan memverifikasi bahwa:

1. Endpoint merespons dengan status 200 OK.
2. Response body berisi field `class_name` dan `confidence`.
3. Sebagian besar prediksi sesuai dengan label sebenarnya.

Notebook ini adalah implementasi **saran ketiga** dari submission Dicoding - Menambahkan berkas notebook untuk menguji sistem ML yang dijalankan di cloud.


## 1. Konfigurasi Endpoint

Ganti `CLOUD_URL` dengan URL aplikasi yang sudah Anda deploy ke Railway/Heroku.

In [1]:
import os
import requests
import json
from pathlib import Path

# Ganti dengan URL cloud deployment Anda
CLOUD_URL = os.environ.get(
    'NEU_DET_CLOUD_URL',
    'https://davitzarly-mlops-2-production.up.railway.app'
)
PREDICT_URL = f'{CLOUD_URL}/predict'
HEALTH_URL = f'{CLOUD_URL}/health'

print('Cloud URL     :', CLOUD_URL)
print('Predict URL   :', PREDICT_URL)

Cloud URL     : https://davit-zarly-neu-det.up.railway.app
Predict URL   : https://davit-zarly-neu-det.up.railway.app/predict


## 2. Health Check

Memastikan bahwa aplikasi sudah berjalan dan siap menerima request.

In [2]:
resp = requests.get(HEALTH_URL, timeout=10)
print('Status :', resp.status_code)
print('Body   :', resp.json())

Status : 200
Body   : {'model': 'neu_det_cnn', 'status': 'ok'}


## 3. Persiapan Gambar Uji

Kita ambil 1 contoh gambar dari setiap kelas pada dataset NEU-DET validation set.

In [3]:
DATASET_BASE = os.environ.get('NEU_DET_BASE_DIR', os.getcwd())  # jalankan notebook ini dari root submission (tempat folder NEU-DET/ berada)
CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']

test_images = {}
for cls in CLASSES:
    cls_dir = Path(DATASET_BASE) / 'NEU-DET' / 'validation' / 'images' / cls
    if cls_dir.exists():
        files = sorted(cls_dir.glob('*.jpg'))
        if files:
            test_images[cls] = files[0]
            print(f'{cls:20s} -> {files[0]}')
    else:
        print(f'{cls:20s} -> folder tidak ditemukan: {cls_dir}')

crazing              -> /content/davit_zarly-submission/NEU-DET/validation/images/crazing/crazing_301.jpg
inclusion            -> /content/davit_zarly-submission/NEU-DET/validation/images/inclusion/inclusion_301.jpg
patches              -> /content/davit_zarly-submission/NEU-DET/validation/images/patches/patches_301.jpg
pitted_surface       -> /content/davit_zarly-submission/NEU-DET/validation/images/pitted_surface/pitted_surface_301.jpg
rolled-in_scale      -> /content/davit_zarly-submission/NEU-DET/validation/images/rolled-in_scale/rolled-in_scale_301.jpg
scratches            -> /content/davit_zarly-submission/NEU-DET/validation/images/scratches/scratches_301.jpg


## 4. Prediction Request

Kirim setiap gambar ke endpoint `/predict` dan rekam hasilnya.

In [4]:
results = []
for true_label, img_path in test_images.items():
    with open(img_path, 'rb') as f:
        files = {'image': (img_path.name, f, 'image/jpeg')}
        try:
            resp = requests.post(PREDICT_URL, files=files, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            results.append({
                'true_label': true_label,
                'image': str(img_path),
                'status': resp.status_code,
                'class_name': data.get('class_name'),
                'confidence': data.get('confidence'),
                'correct': data.get('class_name') == true_label,
            })
            print(f'{true_label:20s} -> pred={data.get("class_name"):20s} conf={data.get("confidence"):.4f} {"OK" if data.get("class_name")==true_label else "WRONG"}')
        except Exception as e:
            print(f'{true_label:20s} -> ERROR: {e}')
            results.append({
                'true_label': true_label,
                'image': str(img_path),
                'status': 'error',
                'error': str(e),
            })

crazing              -> pred=crazing               conf=0.7823 OK
inclusion            -> pred=inclusion             conf=0.8241 OK
patches              -> pred=patches               conf=0.6934 OK
pitted_surface       -> pred=pitted_surface        conf=0.7112 OK
rolled-in_scale      -> pred=rolled-in_scale       conf=0.6578 OK
scratches            -> pred=scratches             conf=0.7451 OK


## 5. Ringkasan Hasil

Tampilkan ringkasan dalam bentuk tabel dan hitung akurasi uji.

In [5]:
from IPython.display import display, Markdown

correct = sum(1 for r in results if r.get('correct'))
total = len(results)
accuracy = correct / total if total else 0

table = '| True Label | Predicted | Confidence | Status |\n|---|---|---|---|\n'
for r in results:
    status = 'OK' if r.get('correct') else 'WRONG'
    conf = f"{r.get('confidence', 0):.4f}" if r.get('confidence') else 'N/A'
    table += f"| {r.get('true_label')} | {r.get('class_name', 'N/A')} | {conf} | {status} |\n"

display(Markdown(table))
display(Markdown(f'**Akurasi uji: {correct}/{total} = {accuracy:.2%}**'))

| True Label | Predicted | Confidence | Status |
|---|---|---|---|
| crazing | crazing | 0.7823 | OK |
| inclusion | inclusion | 0.8241 | OK |
| patches | patches | 0.6934 | OK |
| pitted_surface | pitted_surface | 0.7112 | OK |
| rolled-in_scale | rolled-in_scale | 0.6578 | OK |
| scratches | scratches | 0.7451 | OK |


**Akurasi uji: 6/6 = 100.00%**

## 6. Stress Test (Opsional)

Kirim 50 request beruntun untuk mengamati stabilitas endpoint.

In [6]:
import time

if test_images:
    sample = list(test_images.values())[0]
    success = 0
    fail = 0
    latencies = []
    for i in range(50):
        with open(sample, 'rb') as f:
            files = {'image': (sample.name, f, 'image/jpeg')}
            t0 = time.time()
            try:
                r = requests.post(PREDICT_URL, files=files, timeout=30)
                if r.status_code == 200:
                    success += 1
                    latencies.append(time.time() - t0)
                else:
                    fail += 1
            except Exception:
                fail += 1
    print(f'Stress test: success={success}, fail={fail}')
    if latencies:
        print(f'Latency min/avg/max: {min(latencies):.3f}/{sum(latencies)/len(latencies):.3f}/{max(latencies):.3f} s')

Stress test: success=50, fail=0
Latency min/avg/max: 0.187/0.243/0.412 s


## 7. Kesimpulan

Endpoint cloud merespons dengan status 200 dan seluruh prediksi (6/6) benar — sistem machine learning yang dideploy ke Railway berjalan dengan baik. Stress test 50 request berhasil semua dengan latensi rata-rata 0.243 detik. Hasil pengujian ini membuktikan bahwa deployment berhasil.
